# 第十一章：永续合约 — 加密货币的独特发明 / Chapter 11: Perpetual Swaps — A Crypto-Native Invention

## 你将学到 / What You Will Learn
- 什么是永续合约（和普通期货有什么不同）/ What a perpetual is (and how it differs from a regular future)
- "资金费率"机制是怎么回事 / How the funding-rate mechanism works
- 谁付钱给谁？什么时候付？ / Who pays whom, and when
- 长期持仓的隐性成本 / The hidden cost of holding long-term

---

## 11.1 从普通期货说起 / 11.1 Starting From Ordinary Futures

上一章学的期货有一个特点：**有到期日**。

The futures we met earlier have one defining trait: **a fixed expiry date**.

永续合约 = **没有到期日的期货**。

A perpetual swap = **a future with no expiry**.

怎么保证合约价格不偏离现货太远？答案就是**资金费率(Funding Rate)**。

How is the contract kept anchored to spot? Through the **funding rate**.

## 11.2 资金费率 / 11.2 The Funding Rate

| 情况 / Situation | 谁付钱？ / Who Pays |
|------|---------|
| 合约价 > 现货价 / Contract > spot | **多头付钱给空头** / **Longs pay shorts** |
| 合约价 < 现货价 / Contract < spot | **空头付钱给多头** / **Shorts pay longs** |

- **每8小时结算一次**（一天3次）/ **Settled every 8 hours** (three times per day)
- 费率通常在 -0.75% ~ +0.75% 之间 / The rate is usually capped within −0.75 % to +0.75 %
- 正常市场约 0.01%（很小，但累积起来不少!）/ In calm markets around 0.01 % — tiny per period, but it compounds!

## 11.3 你付的费 = 仓位价值 × 资金费率 / 11.3 What You Pay = Position Value × Funding Rate

> 1 BTC 多头，BTC=$60,000，费率0.05%
> 每8h: $60,000 × 0.05% = $30 → 一天 $90 → 一个月 ≈ $2,700
>
> 1 BTC long at BTC = $60,000, funding 0.05 %.
> Per 8h: $60,000 × 0.05 % = $30 → per day $90 → per month ≈ $2,700.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 11.4 互动实验 / 11.4 Interactive Experiment

In [ ]:
def perpetuals(spot=60000, premium_pct=0.5, position_btc=1.0):
    plt.close('all')
    mark = spot * (1 + premium_pct / 100)
    premium_idx = (mark - spot) / spot * 100
    funding = np.clip(0.01 + np.clip(premium_idx, -0.5, 0.5), -0.75, 0.75)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))

    # Left: funding rate mechanism
    ax = axes[0]
    p_range = np.linspace(-3, 3, 200)
    f_curve = np.clip(0.01 + np.clip(p_range, -0.5, 0.5), -0.75, 0.75)
    ax.plot(p_range, f_curve, color=C_BLUE, lw=2.5)
    ax.fill_between(p_range, f_curve, 0, where=(f_curve > 0), color=C_RED, alpha=0.15, label='Longs pay')
    ax.fill_between(p_range, f_curve, 0, where=(f_curve <= 0), color=C_GREEN, alpha=0.15, label='Shorts pay')
    ax.axhline(y=0, color='gray', ls='--')
    ax.axvline(x=premium_pct, color=C_ORANGE, ls='--', alpha=0.7, lw=2)
    ax.plot(premium_pct, funding, 'ro', ms=12, zorder=5)
    ax.annotate(f'Funding\n{funding:.4f}%', xy=(premium_pct, funding),
                xytext=(premium_pct + 0.6, funding + 0.15),
                fontsize=10, fontweight='bold', color=C_RED,
                arrowprops=dict(arrowstyle='->', color=C_RED, lw=2))
    ax.set_xlabel('Premium (%)', fontsize=11)
    ax.set_ylabel('Funding Rate (%)', fontsize=11)
    ax.set_title('Funding Rate Mechanism', fontsize=14, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

    # Middle: 30-day cumulative
    ax = axes[1]
    hours = np.arange(0, 24 * 30 + 1, 8)
    pos_value = position_btc * mark
    per_charge = pos_value * funding / 100
    cum_long = np.cumsum(np.full(len(hours), per_charge))
    cum_long[0] = 0
    ax.plot(hours / 24, cum_long, color=C_RED, lw=2.5, label='Long cumulative')
    ax.plot(hours / 24, -cum_long, color=C_GREEN, lw=2.5, label='Short cumulative')
    ax.fill_between(hours / 24, cum_long, 0, color=C_RED, alpha=0.1)
    ax.fill_between(hours / 24, -cum_long, 0, color=C_GREEN, alpha=0.1)
    ax.axhline(y=0, color='gray', ls='--')
    ax.set_xlabel('Days', fontsize=11)
    ax.set_ylabel('Cumulative Cost (USD)', fontsize=11)
    ax.set_title(f'30-Day Cost ({position_btc} BTC)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

    # Right: 3-instrument comparison
    ax = axes[2]
    days = np.arange(0, 91)
    spot_path = spot + 500 * np.sin(days / 15)
    perp_path = spot_path * (1 + premium_pct / 100 * np.exp(-days / 30))
    fut_path = spot_path * np.exp(0.05 * (90 - days) / 365)
    ax.plot(days, spot_path, 'k-', lw=2, label='Spot')
    ax.plot(days, perp_path, color=C_BLUE, ls='--', lw=2, label='Perpetual')
    ax.plot(days, fut_path, color=C_ORANGE, ls='-.', lw=2, label='Quarterly Future')
    ax.axvline(x=90, color=C_RED, ls=':', alpha=0.5, label='Future expiry')
    ax.set_xlabel('Day', fontsize=11)
    ax.set_ylabel('Price (USD)', fontsize=11)
    ax.set_title('Perp vs Future vs Spot', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    print(f'  Spot ${spot:,} | Mark ${mark:,.0f} | Premium {premium_pct:.2f}%')
    print(f'  Funding = {funding:.4f}% (per 8h)')
    print(f'  {position_btc} BTC long: per 8h ${per_charge:,.2f} | per day ${per_charge * 3:,.2f} | per month ~${per_charge * 3 * 30:,.0f}')
    if abs(per_charge * 3 * 30) > pos_value * 0.01:
        print('  !! Monthly cost > 1% of position; long-term holding is expensive!')

interact(perpetuals,
         spot=IntSlider(value=60000, min=20000, max=100000, step=1000, description='BTC spot($):'),
         premium_pct=FloatSlider(value=0.5, min=-2, max=2, step=0.1, description='Premium(%):'),
         position_btc=FloatSlider(value=1.0, min=0.1, max=10, step=0.1, description='Pos (BTC):'));

## 11.5 小结 / 11.5 Chapter Recap

| 概念 / Concept | 一句话 / One-liner |
|------|--------|
| 永续合约 / Perpetual swap | 没有到期日的期货，加密货币独有 / Futures with no expiry — unique to crypto |
| 资金费率 / Funding rate | 多空之间的"平衡费"，每8小时收一次 / The balancing fee between longs and shorts, charged every 8 hours |
| 正费率 / Positive funding | 合约贵了，多头付钱给空头 / Contract trades rich; longs pay shorts |
| 负费率 / Negative funding | 合约便宜了，空头付钱给多头 / Contract trades cheap; shorts pay longs |
| 隐性成本 / Hidden cost | 长期持仓的资金费率累积可能很大! / Funding compounds massively over long holds! |

> **下一章：U本位 vs 币本位** —— 两种保证金模式，风险天差地别
>
> **Next chapter: USD-margined vs. coin-margined** — two collateral models with wildly different risk profiles.